In [1]:
from quality_assessment import QualityAssessmentPipeline

pipeline = QualityAssessmentPipeline()

# Run full pipeline
df_result, metrics, features = pipeline.run_full_pipeline(
    '../data/raw/titanic/train.csv'
)

# Display results
print('Quality Metrics:')
print(pipeline.metrics_computer.get_metrics_report(metrics))

print('\nFeatures Summary:')
print(pipeline.feature_engineer.get_features_summary(
    features['column_features'],
    features['dataset_features']
))

# Show some results
print('\nSample Results (first 5 rows):')
df_result.select('row_completeness', 'row_accuracy', 'is_duplicate', 'row_timeliness', 'null_count', 'valid_types_count').show(5)

pipeline.stop()


2026-01-01 20:58:17,104 - DataLoader - INFO - Creating Spark session...
2026-01-01 20:58:17,662 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 20:58:17,662 - DataLoader - INFO - Master: local
2026-01-01 20:58:17,664 - DataLoader - INFO - DataLoader initialized
2026-01-01 20:58:17,664 - DataLoader - INFO - Loading file: ../data/raw/titanic/train.csv
2026-01-01 20:58:17,666 - CSVParser - INFO - Parsing CSV: ../data/raw/titanic/train.csv
2026-01-01 20:58:20,675 - CSVParser - INFO - Loaded 891 rows from ../data/raw/titanic/train.csv
2026-01-01 20:58:20,676 - DataLoader - INFO - Running validation...
2026-01-01 20:58:20,677 - DataValidator - INFO - Running all validations...
2026-01-01 20:58:20,678 - DataValidator - INFO - Validating nulls...
2026-01-01 20:58:21,842 - DataValidator - WARNING - Cabin: 77.1% nulls
2026-01-01 20:58:21,948 - DataValidator - INFO - Validating duplicates...
2026-01-01 20:58:22,415 - DataValidator - INFO - Validating schema...
2026-01-01 20:58:22,417 - Data

Quality Metrics:

DATA QUALITY METRICS REPORT

COMPLETENESS
----------------------------------------
Dataset Level: 91.9%
Column Level:
  PassengerId: 100.0%
  Survived: 100.0%
  Pclass: 100.0%
  Name: 100.0%
  Sex: 100.0%
  Age: 80.13%
  SibSp: 100.0%
  Parch: 100.0%
  Ticket: 100.0%
  Fare: 100.0%
  Cabin: 22.9%
  Embarked: 99.78%

ACCURACY
----------------------------------------
Dataset Level: 91.9%
Column Level:
  PassengerId: 100.0%
  Survived: 100.0%
  Pclass: 100.0%
  Name: 100.0%
  Sex: 100.0%
  Age: 80.13%
  SibSp: 100.0%
  Parch: 100.0%
  Ticket: 100.0%
  Fare: 100.0%
  Cabin: 22.9%
  Embarked: 99.78%

CONSISTENCY
----------------------------------------
Dataset Level: 100.0%
Column Level:
  PassengerId: 100.0%
  Survived: 0.22%
  Pclass: 0.34%
  Name: 100.0%
  Sex: 0.22%
  Age: 9.99%
  SibSp: 0.79%
  Parch: 0.79%
  Ticket: 76.43%
  Fare: 27.83%
  Cabin: 16.61%
  Embarked: 0.45%

TIMELINESS
----------------------------------------
Dataset Level: 100.0%
Column Level:
  Passen

2026-01-01 20:58:31,540 - DataLoader - INFO - Spark session stopped


In [2]:
"""
Unit tests for quality assessment pipeline.
Tests quality metrics, feature engineering, and pipeline integration.
"""

import pytest
from pathlib import Path
from pyspark.sql import SparkSession

from quality_assessment import QualityAssessmentPipeline, QualityMetricsComputer


@pytest.fixture(scope='session')
def spark():
    """Create Spark session for tests."""
    spark = SparkSession.builder \
        .appName('test_quality') \
        .master('local') \
        .config('spark.sql.shuffle.partitions', 4) \
        .getOrCreate()
    
    yield spark
    spark.stop()


@pytest.fixture(scope='session')
def pipeline(spark):
    """Create pipeline for tests."""
    return QualityAssessmentPipeline(spark)


class TestQualityMetrics:
    """Test quality metrics computation."""
    
    def test_completeness_metric(self, pipeline):
        """Test completeness metric."""
        csv_path = 'data/raw/titanic/train.csv'
        
        if Path(csv_path).exists():
            df = pipeline.data_loader.load_file(csv_path)
            completeness = pipeline.metrics_computer.compute_all_metrics(df)
            
            assert 'completeness' in completeness
            assert completeness['completeness']['dataset_level'] >= 0
            assert completeness['completeness']['dataset_level'] <= 100
    
    def test_accuracy_metric(self, pipeline):
        """Test accuracy metric."""
        csv_path = 'data/raw/titanic/train.csv'
        
        if Path(csv_path).exists():
            df = pipeline.data_loader.load_file(csv_path)
            accuracy = pipeline.metrics_computer.compute_all_metrics(df)
            
            assert 'accuracy' in accuracy
            assert accuracy['accuracy']['dataset_level'] >= 0
    
    def test_consistency_metric(self, pipeline):
        """Test consistency metric."""
        csv_path = 'data/raw/titanic/train.csv'
        
        if Path(csv_path).exists():
            df = pipeline.data_loader.load_file(csv_path)
            consistency = pipeline.metrics_computer.compute_all_metrics(df)
            
            assert 'consistency' in consistency
            assert consistency['consistency']['dataset_level'] >= 0


class TestFeatureEngineering:
    """Test feature engineering."""
    
    def test_row_level_features(self, pipeline):
        """Test row-level feature extraction."""
        csv_path = 'data/raw/titanic/train.csv'
        
        if Path(csv_path).exists():
            df = pipeline.data_loader.load_file(csv_path)
            df_features, _, _ = pipeline.feature_engineer.engineer_all_features(df)
            
            assert 'null_count' in df_features.columns
            assert df_features.count() > 0
    
    def test_dataset_level_features(self, pipeline):
        """Test dataset-level feature extraction."""
        csv_path = 'data/raw/titanic/train.csv'
        
        if Path(csv_path).exists():
            df = pipeline.data_loader.load_file(csv_path)
            _, _, dataset_features = pipeline.feature_engineer.engineer_all_features(df)
            
            assert 'row_count' in dataset_features
            assert 'column_count' in dataset_features


class TestFullPipeline:
    """Test full pipeline integration."""
    
    def test_full_pipeline_execution(self, pipeline):
        """Test complete pipeline execution."""
        csv_path = 'data/raw/titanic/train.csv'
        
        if Path(csv_path).exists():
            df_result, metrics, features = \
                pipeline.run_full_pipeline(csv_path, validate=False)
            
            assert df_result.count() > 0
            assert 'completeness' in metrics
            assert 'column_features' in features
    
    def test_quick_assessment(self, pipeline):
        """Test quick assessment."""
        csv_path = 'data/raw/titanic/train.csv'
        
        if Path(csv_path).exists():
            metrics = pipeline.run_quick_assessment(csv_path)
            
            assert 'completeness' in metrics
            assert 'accuracy' in metrics
            assert 'consistency' in metrics


In [4]:
from quality_assessment import QualityAssessmentPipeline

pipeline = QualityAssessmentPipeline()

# Quick assessment (no full pipeline)
metrics = pipeline.run_quick_assessment('../data/raw/hr_analytics/HR_Employee_Attrition_Data.csv')

print(f'Completeness: {metrics["completeness"]["dataset_level"]}%')
print(f'Accuracy: {metrics["accuracy"]["dataset_level"]}%')
print(f'Consistency: {metrics["consistency"]["dataset_level"]}%')
print(f'Timeliness: {metrics["timeliness"]["dataset_level"]}')

pipeline.stop()


2026-01-01 21:00:08,160 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:00:08,160 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:00:08,160 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:00:08,164 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:00:08,164 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:00:08,164 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:00:08,166 - DataLoader - INFO - Master: local
2026-01-01 21:00:08,166 - DataLoader - INFO - Master: local
2026-01-01 21:00:08,166 - DataLoader - INFO - Master: local
2026-01-01 21:00:08,168 - DataLoader - INFO - DataLoader initialized
2026-01-01 21:00:08,168 - DataLoader - INFO - DataLoader initialized
2026-01-01 21:00:08,168 - DataLoader - INFO - DataLoader initialized
2026-01-01 21:00:08,169 - DataLoader - INFO - Loading file: ../data/raw/hr_analytics/HR_Employee_Attrition_Data.csv
2026-01-01 21:00:08,169 - DataLoader - INFO - Loading file: ../data/raw/hr_analy

Completeness: 98.08%
Accuracy: 98.08%
Consistency: 100.0%
Timeliness: 100.0


2026-01-01 21:00:18,459 - DataLoader - INFO - Spark session stopped
2026-01-01 21:00:18,459 - DataLoader - INFO - Spark session stopped
2026-01-01 21:00:18,459 - DataLoader - INFO - Spark session stopped


In [6]:
from quality_assessment import QualityAssessmentPipeline

pipeline = QualityAssessmentPipeline()
df = pipeline.data_loader.load_file('../data/raw/titanic/train.csv')
metrics = pipeline.metrics_computer.compute_all_metrics(df)

print('Null Percentage by Column:')
for col, pct in metrics['completeness']['column_level'].items():
    print(f'  {col}: {pct}%')

print('\nCardinality by Column:')
# FIXED: Unpack the tuple into 3 variables
df_with_features, col_features, dataset_features = pipeline.feature_engineer.engineer_all_features(df)

for col, card in col_features['cardinality'].items():
    print(f'  {col}: {card}%')

print('\nDataset-Level Features:')
for key, value in dataset_features.items():
    print(f'  {key}: {value}')

pipeline.stop()


2026-01-01 21:02:17,650 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:02:17,650 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:02:17,650 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:02:17,650 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:02:17,650 - DataLoader - INFO - Creating Spark session...
2026-01-01 21:02:17,654 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:02:17,654 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:02:17,654 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:02:17,654 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:02:17,654 - DataLoader - INFO - Spark version: 3.5.0
2026-01-01 21:02:17,656 - DataLoader - INFO - Master: local
2026-01-01 21:02:17,656 - DataLoader - INFO - Master: local
2026-01-01 21:02:17,656 - DataLoader - INFO - Master: local
2026-01-01 21:02:17,656 - DataLoader - INFO - Master: local
2026-01-01 21:02:17,656 - DataLoader - INFO - Master: local
2026-

Null Percentage by Column:
  PassengerId: 100.0%
  Survived: 100.0%
  Pclass: 100.0%
  Name: 100.0%
  Sex: 100.0%
  Age: 80.13%
  SibSp: 100.0%
  Parch: 100.0%
  Ticket: 100.0%
  Fare: 100.0%
  Cabin: 22.9%
  Embarked: 99.78%

Cardinality by Column:
  PassengerId: 100.0%
  Survived: 0.22%
  Pclass: 0.34%
  Name: 100.0%
  Sex: 0.22%
  Age: 9.99%
  SibSp: 0.79%
  Parch: 0.79%
  Ticket: 76.43%
  Fare: 27.83%
  Cabin: 16.61%
  Embarked: 0.45%

Dataset-Level Features:
  row_count: 891
  column_count: 12
  memory_usage: 0.08 MB
  data_density: 91.9


2026-01-01 21:02:23,163 - DataLoader - INFO - Spark session stopped
2026-01-01 21:02:23,163 - DataLoader - INFO - Spark session stopped
2026-01-01 21:02:23,163 - DataLoader - INFO - Spark session stopped
2026-01-01 21:02:23,163 - DataLoader - INFO - Spark session stopped
2026-01-01 21:02:23,163 - DataLoader - INFO - Spark session stopped


In [7]:
df_with_features, col_features, dataset_features = pipeline.feature_engineer.engineer_all_features(df)

# Now use each separately
print(col_features['cardinality'])
print(dataset_features['row_count'])


AssertionError: 